# Fine-tuning Lapa or Mamay LLM for Ukrainian Dialect Translation

This notebook fine-tunes [Lapa LLM v0.1.2 Instruct](https://huggingface.co/lapa-llm/lapa-v0.1.2-instruct) or [MamayLM](https://huggingface.co/INSAIT-Institute/MamayLM-Gemma-3-4B-IT-v1.0) on the `merged.csv` dataset for translating Ukrainian dialects to Standard Literary Ukrainian.

**Hardware**: Local NVIDIA RTX 4090 (24 GB) — sufficient for QLoRA 4-bit fine-tuning of 12B models via Unsloth. No cloud GPU (Vast.ai) needed.

**Model**: Lapa LLM (12B, Gemma-3-12B) with optimized Ukrainian tokenizer, or MamayLM (4B, Gemma-3-4B).

**Task**: Translate dialectal Ukrainian text (Hutsul, Boyko, etc.) to Standard Literary Ukrainian.

## Load LLM Model

Using Unsloth's `FastModel` to load the model with 4-bit quantization for memory efficiency.
Fits on a single RTX 4090 (24 GB) with ~9 GB reserved for weights, leaving ~14 GB for training.

In [1]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

from unsloth import FastModel
import torch

MODEL_NAME = "INSAIT-Institute/MamayLM-Gemma-3-4B-IT-v1.0"
MAX_SEQ_LENGTH = 512  # longer context for dialect sentences + instruction template

model, tokenizer = FastModel.from_pretrained(
    model_name = MODEL_NAME,
    max_seq_length = MAX_SEQ_LENGTH,
    load_in_4bit = True,
    dtype = None,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.1.4: Fast Gemma3 patching. Transformers: 4.57.1.
   \\   /|    NVIDIA GeForce RTX 4090. Num GPUs = 1. Max memory: 23.518 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 8.9. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.29.post3. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Gemma3 does not support SDPA - switching to fast eager.


model.safetensors.index.json: 0.00B [00:00, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.96G [00:00<?, ?B/s]

Cancellation requested; stopping current tasks.


KeyboardInterrupt: 

## Prepare LoRA Configuration

Configure LoRA (Low-Rank Adaptation) for efficient fine-tuning. This allows updating only a small subset of parameters.

In [ ]:
model = FastModel.get_peft_model(
    model,
    r = 2,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 2,          # same as r for stable scaling
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 42,
    use_rslora = False,
    loftq_config = None,
)

## Load and Prepare Dataset

Load the `merged.csv` dataset and format it for instruction fine-tuning.

In [ ]:
import pandas as pd
from datasets import Dataset

# Load train/val split
train_df = pd.read_csv('../../data/parallel/train.csv')
val_df   = pd.read_csv('../../data/parallel/val.csv')

print(f"Train size: {len(train_df)} samples")
print(f"Val size:   {len(val_df)} samples")
print("\nColumns:", train_df.columns.tolist())
print("\nFirst few examples:")
print(train_df.head())

In [ ]:
def format_prompt(dialect_text, standard_text):
    return {
        "text": f"""<start_of_turn>user
Переклади наступний діалектний текст на стандартну літературну українську мову:

{dialect_text}<end_of_turn>
<start_of_turn>model
{standard_text}<end_of_turn>"""
    }

train_dataset = Dataset.from_list([
    format_prompt(row['source'], row['target'])
    for _, row in train_df.iterrows()
])

val_dataset = Dataset.from_list([
    format_prompt(row['source'], row['target'])
    for _, row in val_df.iterrows()
])

print(f"Train dataset: {len(train_dataset)} samples")
print(f"Val dataset:   {len(val_dataset)} samples")
print("\nExample:")
print(train_dataset[0]['text'])

## Configure Training

Set up training parameters using Unsloth's optimized trainer.

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_dataset,
    eval_dataset = val_dataset,
    dataset_text_field = "text",
    max_seq_length = MAX_SEQ_LENGTH,
    dataset_num_proc = 2,
    packing = True,            # fills context window with multiple short samples — ~5x more efficient
    args = TrainingArguments(
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 4,
        warmup_steps = 50,
        num_train_epochs = 3,
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 50,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "cosine",
        seed = 42,
        output_dir = "outputs",
        report_to = "none",
        eval_strategy = "steps",
        eval_steps = 1000,
        save_strategy = "steps",
        save_steps = 1000,
        load_best_model_at_end = True,
        metric_for_best_model = "eval_loss",
    ),
)

## Train the Model

Start fine-tuning. This may take a while depending on dataset size and GPU.

In [ ]:
# Show GPU memory before training
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

GPU = NVIDIA GeForce RTX 4090. Max memory = 23.527 GB.
8.74 GB of memory reserved.


In [ ]:
# Train!
trainer_stats = trainer.train()

In [ ]:
# Show memory usage after training
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

## Test the Model

Try inference with the fine-tuned model.

In [ ]:
# Enable fast inference mode
FastModel.for_inference(model)

# Test with a dialectal example
test_dialect = "Мій любимий празник новий год і дєнь раждєнія"  # Replace with actual test text

inputs = tokenizer(
    f"""<start_of_turn>user
Переклади наступний діалектний текст на стандартну літературну українську мову:

{test_dialect}<end_of_turn>
<start_of_turn>model
""",
    return_tensors="pt"
).to("cuda")

outputs = model.generate(
    **inputs,
    max_new_tokens=256,
    temperature=0.7,
    top_p=0.9,
    use_cache=True
)

result = tokenizer.decode(outputs[0], skip_special_tokens=False)
print("\n=== Translation Result ===")
print(result)

## Save the Model

Save LoRA adapters, upload to HuggingFace, or export in various formats.

### Save LoRA Adapters Locally

In [ ]:
# Save LoRA adapters (lightweight, ~200MB)
model.save_pretrained("mama_dialect_lora")
tokenizer.save_pretrained("mama_dialect_lora")
print("Saved LoRA adapters to 'mama_dialect_lora'")

### Upload to HuggingFace Hub

In [ ]:
# Upload to HuggingFace (requires token)
if False:  # Set to True to upload
    HF_TOKEN = "YOUR_HF_TOKEN"
    HF_REPO = "YOUR_USERNAME/lapa-dialect-translation-lora"

    model.push_to_hub(HF_REPO, token=HF_TOKEN)
    tokenizer.push_to_hub(HF_REPO, token=HF_TOKEN)
    print(f"Pushed to {HF_REPO}")

### Save Merged Model (Full FP16)

In [ ]:
# Save merged model for deployment (requires more disk space)
if False:  # Set to True to save merged model
    model.save_pretrained_merged("lapa_dialect_full", tokenizer)
    print("Saved merged model to 'lapa_dialect_full'")

### Save as GGUF for llama.cpp

In [ ]:
# Save as GGUF for llama.cpp deployment
if False:  # Set to True to save GGUF
    model.save_pretrained_gguf(
        "lapa_dialect_gguf",
        tokenizer,
        quantization_method="q4_k_m",  # Good balance of size/quality
    )
    print("Saved GGUF model to 'lapa_dialect_gguf'")

## Load Saved Model

To load the saved LoRA adapters for inference:

In [ ]:
if False:  # Set to True to load saved model
    from unsloth import FastModel

    model, tokenizer = FastModel.from_pretrained(
        model_name="lapa_dialect_lora",
        max_seq_length=2048,
        load_in_4bit=True,
    )
    FastModel.for_inference(model)
    print("Loaded model from 'lapa_dialect_lora'")